# VQAv2 Tokenization A/B Study with LLaVA-1.5

This notebook:

- Loads LLaVA-1.5 (Hugging Face) via the repo's VLM loader
- Samples VQAv2 validation questions
- Selects critical entities in each question via POS tagging (spaCy)
- Enforces minimal interventions that change the SAME entity from 1 token (A) to 2 tokens (B)
- Evaluates each intervention separately to measure A vs B performance disparity
- Verifies that interventions actually flip tokenization as intended

Key requirement: For each question, the same entity must tokenize as 1 token in case A and 2 tokens in case B via minimal prompt modifications.


In [ ]:
# Install spaCy English model if needed
import sys
import subprocess


def pip_install(pkg: str) -> None:
    print(f"Installing {pkg}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])


try:
    import spacy  # type: ignore
except Exception:
    pip_install("spacy==3.7.3")
    import spacy  # type: ignore

try:
    nlp = spacy.load("en_core_web_sm")
except Exception:
    pip_install(
        "https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl"
    )
    import en_core_web_sm  # type: ignore

    nlp = en_core_web_sm.load()

print("spaCy model loaded:", nlp)


In [ ]:
# Imports from this repo and base libs
from pathlib import Path
import json
import random
from typing import List, Tuple, Dict, Any, Optional

import torch
from PIL import Image

# Use local repo utilities
from vlm_eval.models import load_vlm
from vlm_eval.conf import DatasetConfig, DatasetRegistry
from vlm_eval.tasks.harnesses.vqav2 import VQAv2IndexDataset

# Reproducibility
random.seed(21)
torch.manual_seed(21)

# Paths from this repo's configuration
DATA_ROOT = Path("/localdisk/ssrivas9/vlm-evaluation")
VQAV2_META = Path("datasets/vqa-v2/metadata-slim-1024.json")
IMG_ROOT = DATA_ROOT

# HF token handling: either env var or .hf_token file in repo root
import os
HF_TOKEN = None
if (DATA_ROOT / ".hf_token").exists():
    HF_TOKEN = (DATA_ROOT / ".hf_token").read_text().strip()
else:
    HF_TOKEN = os.environ.get("HF_TOKEN")

print("Using HF token:", "yes" if HF_TOKEN else "no")


In [ ]:
# Load a small index dataset from VQAv2
index_dataset = VQAv2IndexDataset(DATA_ROOT, VQAV2_META)
print(f"Loaded {len(index_dataset)} VQAv2 examples (slim-1024)")

# Peek a few
for i in range(3):
    qid, question, img_path, answer = index_dataset[i]
    print({"qid": qid, "q": question, "a": answer, "img": str(img_path)})


In [ ]:
# Load LLaVA-1.5 model using repo loader
# We use the official HF hub id via the model family 'llava-v15'

MODEL_FAMILY = "llava-v15"
MODEL_ID = "llava-v1.5-7b"
RUN_DIR = Path("liuhaotian/llava-v1.5-7b")  # hf hub path is accepted by loader

vlm = load_vlm(
    model_family=MODEL_FAMILY,
    model_id=MODEL_ID,
    run_dir=RUN_DIR,
    hf_token=HF_TOKEN,
    load_precision="bf16",
    max_length=128,
    temperature=0.2,
)

prompt_fn = vlm.get_prompt_fn("vqa-v2")
image_processor = vlm.image_processor
print("Loaded VLM:", MODEL_ID)


In [ ]:
from transformers import PreTrainedTokenizerBase, AutoTokenizer

# Access underlying tokenizer
tokenizer: PreTrainedTokenizerBase = vlm.tokenizer  # type: ignore

# Try to get fast tokenizer for offset mapping (fallback gracefully if unavailable)
try:
    tokenizer_fast = AutoTokenizer.from_pretrained(str(RUN_DIR), use_fast=True)
    print("Fast tokenizer available for offset mapping")
except Exception as e:
    tokenizer_fast = None
    print("Fast tokenizer unavailable (will use approximations):", e)

# Helper: get spaCy noun phrases or fallback to nouns/pronouns
from spacy.matcher import Matcher

matcher = Matcher(nlp.vocab)
matcher.add("NOUN_PHRASE", [[{"POS": "DET", "OP": "?"}, {"POS": "ADJ", "OP": "*"}, {"POS": "NOUN"}]])


def find_candidate_entities(text: str, max_len: int = 3) -> List[str]:
    doc = nlp(text)
    spans = []
    # Use noun chunks first
    for chunk in doc.noun_chunks:
        t = chunk.text.strip()
        if 1 <= len(t.split()) <= max_len:
            spans.append(t)
    # Also try simple matcher
    for match_id, start, end in matcher(doc):
        t = doc[start:end].text.strip()
        if 1 <= len(t.split()) <= max_len:
            spans.append(t)
    # Fallback to single nouns/pronouns if nothing
    if not spans:
        spans = [t.text for t in doc if t.pos_ in {"NOUN", "PROPN", "PRON"}]
    # Deduplicate with order preserved
    seen = set()
    uniq = []
    for s in spans:
        if s.lower() not in seen:
            seen.add(s.lower())
            uniq.append(s)
    return uniq


def tokenized_length(s: str) -> int:
    return len(tokenizer.encode(s, add_special_tokens=False))


In [ ]:
# Build A/B with the SAME entity via minimal, local prompt edits, verified in-context

HAIR_SPACE = "\u200a"  # thin space, visually minimal
ZW_SPACE = "\u200b"    # zero-width space (fallback)

# Define minimal interventions that can flip tokenization
TRANSFORMS = [
    ("wrap_quotes", '"', '"'),
    ("wrap_unicode_quotes", "\u201c", "\u201d"),
    ("wrap_parentheses", "(", ")"),
    ("prefix_hair_space", HAIR_SPACE, ""),
    ("prefix_zw_space", ZW_SPACE, ""),
]


def get_offsets(text: str):
    """Get offset mapping from fast tokenizer if available."""
    if tokenizer_fast is None:
        return None, None
    try:
        enc = tokenizer_fast.encode_plus(text, add_special_tokens=False, return_offsets_mapping=True)
        return enc.get("offset_mapping"), enc.get("input_ids")
    except Exception:
        return None, None


def entity_token_len_in_text(text: str, start: int, length: int) -> Optional[int]:
    """Get exact token count for entity at given character position in text."""
    offsets, _ = get_offsets(text)
    if offsets is None:
        return None
    end = start + length
    tok_start = None
    tok_end = None
    for i, (cs, ce) in enumerate(offsets):
        if cs <= start < ce and tok_start is None:
            tok_start = i
        if cs < end <= ce:
            tok_end = i + 1
            break
    if tok_start is None or tok_end is None:
        return None
    return tok_end - tok_start


def find_all_occurrences(text: str, sub: str) -> List[int]:
    """Find all case-insensitive occurrences of substring in text."""
    pos = 0
    locs = []
    while True:
        idx = text.lower().find(sub.lower(), pos)
        if idx == -1:
            break
        locs.append(idx)
        pos = idx + 1
    return locs


def build_wrapped_variant(q: str, ent: str, pos: int, pre: str, post: str) -> Tuple[str, int]:
    """Build variant with prefix/suffix around entity at given position."""
    prefix = q[:pos]
    suffix = q[pos + len(ent):]
    new_q = prefix + pre + ent + post + suffix
    new_start = len(prefix) + len(pre)
    return new_q, new_start


def select_same_entity_local_edits(q: str) -> Dict[str, List[Dict[str, Any]]]:
    """Return dict: intervention_name -> list of {entity, qA, qB} for which A=1 token, B=2 tokens in-context."""
    buckets: Dict[str, List[Dict[str, Any]]] = {name: [] for name, _, _ in TRANSFORMS}
    cands = find_candidate_entities(q)
    
    for ent in cands:
        occs = find_all_occurrences(q, ent)
        for pos in occs:
            # A: original context must be 1 token for this occurrence
            tA = entity_token_len_in_text(q, pos, len(ent))
            if tA is None:  # fallback to isolated tokenization
                tA = tokenized_length(ent)
            if tA != 1:
                continue
                
            # Try minimal local edits for B
            for name, pre, post in TRANSFORMS:
                qb, new_pos = build_wrapped_variant(q, ent, pos, pre, post)
                tB = entity_token_len_in_text(qb, new_pos, len(ent))
                if tB is None:  # fallback to isolated tokenization
                    # For wrapped entities, check the wrapped version
                    wrapped_entity = pre + ent + post
                    tB = tokenized_length(wrapped_entity)
                if tB == 2:
                    buckets[name].append({
                        "entity": ent, 
                        "qA": q, 
                        "qB": qb, 
                        "posA": pos, 
                        "posB": new_pos,
                        "tA": tA,
                        "tB": tB
                    })
    return buckets

print("Defined intervention functions")


In [ ]:
# Aggregate examples per intervention, ensuring each has sufficient samples
per_intervention: Dict[str, List[Dict[str, Any]]] = {name: [] for name, _, _ in TRANSFORMS}

for i in range(len(index_dataset)):
    qid, question, img_path, answer = index_dataset[i]
    found = select_same_entity_local_edits(question)
    
    for name in per_intervention.keys():
        for item in found.get(name, []):
            per_intervention[name].append({
                "qid": qid,
                "q": question,
                "img_path": img_path,
                "answer": answer,
                **item,
            })
    
    # Stop when each intervention has at least 25 examples
    if all(len(v) >= 25 for v in per_intervention.values()):
        break

# Report counts per intervention
for name, items in per_intervention.items():
    print(f"{name} => {len(items)} examples")

# Show a few examples with decoded tokens
def get_entity_tokens_in_context(question: str, entity: str, entity_pos: int) -> Tuple[List[int], List[str], str]:
    """Extract tokens for entity within question context using offset mapping."""
    offsets, token_ids = get_offsets(question)
    
    if offsets is None or token_ids is None:
        # Fallback to isolated tokenization
        tokens = tokenizer.encode(entity, add_special_tokens=False)
        decoded_tokens = [tokenizer.decode([t]) for t in tokens]
        full_decoded = tokenizer.decode(tokens)
        return tokens, decoded_tokens, full_decoded
    
    # Find token span for entity
    entity_end = entity_pos + len(entity)
    start_token = None
    end_token = None
    
    for i, (char_start, char_end) in enumerate(offsets):
        if char_start <= entity_pos < char_end and start_token is None:
            start_token = i
        if char_start < entity_end <= char_end:
            end_token = i + 1
            break
    
    if start_token is not None and end_token is not None:
        entity_tokens = token_ids[start_token:end_token]
        decoded_tokens = [tokenizer.decode([t]) for t in entity_tokens]
        full_decoded = tokenizer.decode(entity_tokens)
        return entity_tokens, decoded_tokens, full_decoded
    else:
        # Fallback
        tokens = tokenizer.encode(entity, add_special_tokens=False)
        decoded_tokens = [tokenizer.decode([t]) for t in tokens]
        full_decoded = tokenizer.decode(tokens)
        return tokens, decoded_tokens, full_decoded


print("\nExample A/B pairs with token details:")
for name, items in per_intervention.items():
    if items:
        ex = items[0]
        entity = ex['entity']
        qA = ex['qA']
        qB = ex['qB']
        posA = ex['posA']
        posB = ex['posB']
        
        print(f"\n{name}:")
        print(f"  Entity: '{entity}'")
        
        # Show 1-token version (Case A) - extract from original question context
        tokens_a, decoded_a, full_a = get_entity_tokens_in_context(qA, entity, posA)
        print(f"  Case A (1 token): {tokens_a} → {decoded_a} → '{full_a}'")
        
        # Show 2-token version (Case B) - extract from modified question context  
        tokens_b, decoded_b, full_b = get_entity_tokens_in_context(qB, entity, posB)
        print(f"  Case B (2 tokens): {tokens_b} → {decoded_b} → '{full_b}'")
        
        print(f"  Question A: '{qA}'")
        print(f"  Question B: '{qB}'")
        
        # Additional verification: show what the intervention actually changed
        for transform_name, pre, post in TRANSFORMS:
            if transform_name == name:
                print(f"  Intervention: '{entity}' → '{pre}{entity}{post}'")
                break
        print()


In [ ]:
# Sanity check: re-verify tokenization flip for each collected pair per intervention
# This ensures interventions actually change tokenization as intended

def verify_tokenization_in_context(q: str, ent: str) -> int:
    """Get token count for entity in context, with fallback to isolated count."""
    offsets, _ = get_offsets(q)
    if offsets is None:
        return tokenized_length(ent)
    
    start = q.lower().find(ent.lower())
    if start == -1:
        return tokenized_length(ent)
    
    end = start + len(ent)
    ts = te = None
    for i, (cs, ce) in enumerate(offsets):
        if cs <= start < ce and ts is None:
            ts = i
        if cs < end <= ce:
            te = i + 1
            break
    return (te - ts) if ts is not None and te is not None else tokenized_length(ent)


sanity_results = {}
for name, rows in per_intervention.items():
    valid_pairs = 0
    invalid_pairs = []
    
    for s in rows:
        qA, qB, ent = s["qA"], s["qB"], s["entity"]
        
        # Re-verify tokenization
        la = verify_tokenization_in_context(qA, ent)
        lb = verify_tokenization_in_context(qB, ent)
        
        if la == 1 and lb == 2:
            valid_pairs += 1
        else:
            invalid_pairs.append({
                "qid": s["qid"], 
                "entity": ent, 
                "la": la, 
                "lb": lb,
                "qA": qA,
                "qB": qB
            })
    
    sanity_results[name] = {
        "valid": valid_pairs,
        "invalid": len(invalid_pairs),
        "invalid_examples": invalid_pairs[:3]  # Show first 3 failures
    }

print("Sanity check results (A=1 token, B=2 tokens):")
for name, result in sanity_results.items():
    print(f"{name}: {result['valid']} valid, {result['invalid']} invalid")
    if result['invalid_examples']:
        print(f"  First invalid example: {result['invalid_examples'][0]}")

# Filter to keep only valid pairs
filtered_per_intervention = {}
for name, rows in per_intervention.items():
    valid_rows = []
    for s in rows:
        qA, qB, ent = s["qA"], s["qB"], s["entity"]
        la = verify_tokenization_in_context(qA, ent)
        lb = verify_tokenization_in_context(qB, ent)
        if la == 1 and lb == 2:
            valid_rows.append(s)
    filtered_per_intervention[name] = valid_rows

print("\nFiltered counts (valid pairs only):")
for name, items in filtered_per_intervention.items():
    print(f"{name} => {len(items)} valid examples")


In [ ]:
# Run the VLM on same-entity A/B prompts, per intervention, and compare outputs
from PIL import Image

per_intervention_results: Dict[str, List[Dict[str, Any]]] = {name: [] for name, _, _ in TRANSFORMS}

print("Running VLM evaluation per intervention...")
for name, items in filtered_per_intervention.items():
    print(f"Processing {name}: {len(items)} examples")
    
    for i, s in enumerate(items):
        if i % 10 == 0:
            print(f"  {i}/{len(items)}")
            
        img = Image.open(s["img_path"]).convert("RGB")
        qA = s["qA"]
        qB = s["qB"]
        promptA = prompt_fn(qA)
        promptB = prompt_fn(qB)

        if hasattr(image_processor, "__call__"):
            pixel_values_single = image_processor(img, return_tensors="pt")["pixel_values"][0]
        else:
            raise RuntimeError("Unexpected image_processor type")

        pixel_values_single = pixel_values_single.to(vlm.distributed_state.device)  # type: ignore
        pixel_values = torch.stack([pixel_values_single, pixel_values_single], dim=0)

        outs = vlm.generate_answer(pixel_values, [promptA, promptB])
        outA, outB = outs[0], outs[1]

        per_intervention_results[name].append({
            "qid": s["qid"],
            "gt": s["answer"],
            "entity": s["entity"],
            "qA": qA,
            "qB": qB,
            "outA": outA,
            "outB": outB,
        })

print("\nCompleted VLM evaluation. Results per intervention:")
print({k: len(v) for k, v in per_intervention_results.items()})


In [ ]:
# Evaluate correctness metrics per intervention: A (1 token) vs B (2 tokens)

def normalize_ans(s: str) -> str:
    return s.strip().lower()


metrics = {}
for name, rows in per_intervention_results.items():
    accA = 0
    accB = 0
    for r in rows:
        gt = normalize_ans(r["gt"])
        accA += normalize_ans(r["outA"]) == gt
        accB += normalize_ans(r["outB"]) == gt
    
    n = len(rows)
    if n > 0:
        acc_a = accA / n
        acc_b = accB / n
        delta = acc_a - acc_b
        metrics[name] = {
            "n": n, 
            "accA": acc_a, 
            "accB": acc_b, 
            "delta_A_minus_B": delta
        }
    else:
        metrics[name] = {"n": 0, "accA": 0.0, "accB": 0.0, "delta_A_minus_B": 0.0}

print("Performance metrics per intervention:")
print("(A = entity as 1 token, B = entity as 2 tokens)")
print()
for name, metric in metrics.items():
    print(f"{name}:")
    print(f"  n={metric['n']}, accA={metric['accA']:.3f}, accB={metric['accB']:.3f}, delta={metric['delta_A_minus_B']:.3f}")
    if metric['delta_A_minus_B'] > 0:
        print(f"  → A (1 token) performs BETTER than B (2 tokens)")
    elif metric['delta_A_minus_B'] < 0:
        print(f"  → B (2 tokens) performs BETTER than A (1 token)")
    else:
        print(f"  → No difference between A and B")
    print()

# Summary
print("Summary:")
total_examples = sum(m['n'] for m in metrics.values())
print(f"Total examples across all interventions: {total_examples}")
significant_deltas = [name for name, m in metrics.items() if abs(m['delta_A_minus_B']) > 0.05]
print(f"Interventions with >5% performance difference: {significant_deltas}")


### Why can a word tokenize to one token or two in context?

- **Byte Pair Encoding (BPE)** and similar subword algorithms segment text into frequent chunks from a learned vocabulary.
- The same surface word can map to different token sequences depending on:
  - **Surrounding characters** (spaces, punctuation, quotes) that change pre-tokenization normalization and word boundaries.
  - **Casing or accents** that break merges.
  - **Whitespace handling**: some tokenizers learn merges that apply only when preceded by a space (e.g., Llama-style leading space rules).
  - **Special tokens or chat templates** that introduce control tokens, which can affect how a following word is segmented.

- **Practically**: if a subword merge exists for "word" but only when preceded by a space, placing it within quotes, parentheses, or after special characters can disable that merge and split it into two tokens.

### This notebook's approach:

We enforce A vs B by applying minimal interventions around the same entity:
- **Quotes**: `word` → `"word"` 
- **Parentheses**: `word` → `(word)`
- **Thin spaces**: `word` → `[thin-space]word`

Each intervention is validated to ensure it actually flips tokenization from 1→2 tokens, then evaluated separately to measure any performance disparity.
